![NWM](../img/NWM.png)

# Analyze National Water Model Outputs vs Observed Snow Water Equivalent (SWE)
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org), Anthony Castronova (acastronova@cuahsi.org)   
Last updated: Mar 27, 2026

#### Introduction:  
This notebook demonstrates how to perform a point-scale analysis comparing modeled and observed SWE at selected CCSS (California Cooperative Snow Surveys) sites.  We focus on analyzing model performance for **single sites**, with particular attention to the **magnitude and timing of peak SWE**.   


## 1. Setup

### 1a. Python Environment

Ensure that the `cssi_evaluation` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions in the `GettingStarted.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys
import glob
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import pyproj
import geopandas as gpd
import pandas as pd
import xarray as xr
from dask.distributed import Client
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from cssi_evaluation.models import nwm_utils
from cssi_evaluation.variables import snow_utils
from cssi_evaluation.utils import plot_utils, dataPrep_utils, evaluation_utils, metric_utils
from cssi_evaluation.external_data_access import observation_utils

hv.extension('bokeh')

%load_ext autoreload
%autoreload 2

### 1b. Dask

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster. You may have to adjust the number of workers and their memory limit to match the capabilities of the system you're running on.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

### 1c. Set Paths and Parameters

In [ ]:
# repository path
repo_root = Path.cwd().resolve().parents[1]

# path to the model domain data
domain_data_path = f"{repo_root}/examples/nwm/domain_data/" 

# Path to the watershed shapefile
watershed_path = f"{domain_data_path}TolumneRiver_18040009.shp"

# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2020-09-30'

# Path to NWM snow data. This is a cloud bucket where NOAA stores operational
# simulation data on AWS. The specific dataset we'll be accessing is a 
# 40-year retrospective simulation.
conus_bucket_url = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/ldasout.zarr'

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs_CCSS' 
MOD_OutputFolder = './mod_outputs_NWM'

## 2. Retrieve Observed Snow Data

This code reads a GeoJSON file of all snow observation stations and filters them to include only stations in California with available CSV data. It also loads the watershed boundary shapefile (`TolumneRiver_18040009.shp`), converts it to the appropriate coordinate system, and merges all watershed polygons into a single MultiPolygon. Finally, it counts how many observation sites fall within this combined watershed boundary, providing the number of sites located inside the study area.

**Heads up**: You might need to run the next cell twice. Sometimes, the spatial filtering does not return any results the first time due to how geometries are loaded or processed in the background.

In [ ]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Read the watershed shapefile and standardize to WGS84
watershed_gdf = gpd.read_file(watershed_path).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')

Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
gdf_in_bbox

In [ ]:
# Plot the sites within the watershed boundary using the plot_utils function.
# This is a helper function provided by the evaluation framework to assist analyses.
plot_utils.map_sites_within_watershed(gdf_in_bbox, watershed_gdf, zoom_start=9) 

Now that we know which data are available within our watershed, we'll use the `observation_utils.getCCSSData` utility function to access them. We'll save these locally and access them later in the notebook for our analysis.

In [ ]:
# Create a folder to save results. Raise an error if the path already exists
Path(OBS_OutputFolder).mkdir(exist_ok=False)

In [ ]:
# Collect data using the `getCCSSData` utility function. This returns a 
# DataFrame object that we can use later and also saves the data to CSV.
obs_df = observation_utils.getCCSSData(gdf_in_bbox, # GeoPandas DataFrame containing sites 
                                       'CA',        # State abbreviation 
                                       StartDate,   # Start date in YYYY-MM-DD format
                                       EndDate,     # End date in YYYY-MM-DD format
                                       f'{OBS_OutputFolder}/obs_ccss_data.csv')


In [ ]:
# preview the data by looking at the first few records.
obs_df.head()

## 3. Retrieve Snow Model Outputs

NOAA shares inputs and outputs to the National Water Model retrospective simulations version 3 at <a href="https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html" style="color: blue; background-color: snow;">https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html</a>. The following code uses `fsspec` and `xarray` Python libraries to load the Zarr metadata of snow outputs (located within **ldasout.zarr**) into memory. Once the code is executed, you can see the wall time, which includes time spent waiting for I/O operations, such as reading data from a remote server. In our case, it took about 12 seconds to load the metadata into memory. Set up `Dask`, a parallel computing library, to enable performing operations on large datasets that don't fit into memory by breaking them into smaller, manageable pieces called chunks.

In [ ]:
%%time 
ds = xr.open_zarr(
    store=conus_bucket_url,
    consolidated=True,      # Tells Xarray to use consolidated metadata (speeds up access)
    storage_options={       
        "anon": True,       # Means anonymous access to the public S3 bucket without needing AWS credentials
        "client_kwargs": {"region_name": "us-east-1"}  # Specifies the AWS region where the bucket lives
    }
)

In [ ]:
# preview the dataset
ds

Download the Snow Water Equivalent predictions from this archive of data by using the `getNWMSWE` utility function.

In [ ]:
# Create a folder to save results. If the folder already exists, an error will be raised.
Path(MOD_OutputFolder).mkdir(exist_ok=False)

# Download NWM SWE data for the sites within the watershed bounding box, save to /mod_outputs folder
input_crs = 'EPSG:4326' # WGS84 lat/lon. This is the CRS of the input geodataframe (gdf_in_bbox) 

network = 'CCSS'        # This is based on the point observation network (e.g., SNOTEL, SCAN, CCSS) that the sites belong to. 
                        # It is not related to the NWM model data, which is the same regardless of the point observation network.

model_df = nwm_utils.getNWMSWE(
    gdf_in_bbox,                            # GeoPandas DataFrame containing sites
    input_crs,                              # Coordinate system of the sites
    network,                                # Network of the observation data
    conus_bucket_url,                       # Zarr url containing the NWM predictions
    StartDate,                              # Start date in YYYY-MM-DD format
    EndDate,                                # End date in YYYY-MM-DD format
    f'{MOD_OutputFolder}/mod_nwm_data.csv'  # Location to save output data
)

In [ ]:
# preview the data we collected by printing the first few records.
model_df.head()

### Quick plot sanity check

Plot a simple timeseries of modeled and observed SWE to make sure our data retrieval was successful.

In [ ]:
# Ensure date columns are in datetime format for both dataframes
obs_df.Date = pd.to_datetime(obs_df.Date)
model_df.Date = pd.to_datetime(model_df.Date)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# plot modeled and observed SWE
ax.plot(model_df["Date"], model_df["TUM:CA:CCSS"], label="Modeled", linewidth=2)
ax.plot(obs_df["Date"], obs_df["TUM:CA:CCSS"], label="Observed", linewidth=2)


# format the plot axis, legend, grid, etc.
ax.set_xlabel("Date")
ax.set_ylabel("SWE (mm)")

# Date formatting for x-axis
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()

## 4. Compare Modeled and Observed SWE Timeseries at a Single Site

Once we have both observation data and modeling outpus, it's important to evaluate how well the model reproduces observed data. The following plots are simple timeseries comparisons of **modeled vs. observed** SWE. These types of plots provide a straight-forward visual of how well the observations and simulations agree and are a great start for assessing general model performance.  

📊 We include two figures:

1. **Time Series Overlay:** Plots the observed and modeled values together over time. This helps identify:
   - Periods of systematic bias
   - Timing differences in peaks and lows
   - General agreement in trends

2. **Scatter Plot with 1:1 Line:** Plots each modeled value against its corresponding observed value. This highlights:
   - Accuracy across the full range of values
   - Over- or under-prediction patterns
   - Outliers or extreme events

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Recall, we also printed out the site metadata for all sites within the watershed, which contains the 3-letter site codes.

✏️ Once you’ve identified the site of interest, **enter its site code in the next code cell for `my_site_code`**:   

In [ ]:
# choose a site of interest within the watershed
my_site_code = 'TUM:CA:CCSS'

In [ ]:
# make sure data is loaded
obs_df = pd.read_csv(f'{OBS_OutputFolder}/obs_ccss_data.csv')
model_df = pd.read_csv(f'{MOD_OutputFolder}/mod_nwm_data.csv')

# make sure date columns are datetime and set as index for easier plotting and metric calculations
obs_df["Date"] = pd.to_datetime(obs_df["Date"])
model_df["Date"] = pd.to_datetime(model_df["Date"])

obs_df = obs_df.set_index("Date")
model_df = model_df.set_index("Date")

Use the `comparison_plots` utility function to provide a quicky view of the data.

In [ ]:
plot_utils.comparison_plots(obs_df,  
                            model_df, 
                            f'{my_site_code}',  # Column name in df_obs
                            f'{my_site_code}',  # Column name in df_model
                            site_label=None)

To move beyond an overall summary of daily performance, we replot the modeled vs. observed SWE scatter while highlighting specific months with a distinct color. This gives us more information about the **seasonal model performance**.  

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization reveals whether there are **seasonal patterns** in the relationship between observed and modeled SWE, allowing us to distinguish model behavior during the key snowpack phases of <span style="color: teal;">accumulation</span> and <span style="color: tomato;">ablation (melt)</span>. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season.

You can change the list of highlighted months (for example, October–December for early accumulation or March–May for spring melt) to explore in the scatter plot how model performance varies across different parts of the snow season. This seasonal perspective motivates the _peak SWE analysis_ that follows.

##### 📊 For this example, let's highlight the _early snow accumulation period_ of October - January:

In [ ]:
plot = plot_utils.plot_custom_scatter_SWE(
    obs_df,                                   # Pandas DataFrame containing observations
    model_df,                                 # Pandas DataFrame containing model predictions
    f"{my_site_code}",                        # Column name corresponding with observation series that will be plotted
    f"{my_site_code}",                        # Column name corresponding with the modeled series that will be plotted
    site_label=my_site_code,                  # Clean name for the plot title
    highlight_months=[10, 11, 12, 1],         # Months to highlight in the plot
)

plot

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>What does this plot tell us about how well the model performs during the early snow accumulation period at this site?<br>
HINT: How close are the <span style="color: teal;">green points</span> to the 1:1 line?</p>
</div>

## 5. Peak SWE Evaluation at the Watershed Scale  
As we saw in the previous section, how well a model matches observations can differ greatly throughout the year. The following section focuses on **peak SWE** (or maximum SWE) analysis. 

**Peak SWE is a key diagnostic for snow-dominated hydrologic systems** because it represents the maximum amount of liquid water stored in the snowpack before the spring melt. Evaluating both the magnitude (quantity) and timing (date) of peak SWE provides insight into whether the model is accurately representing snow accumulation and seasonal energy balance.  

Errors in peak SWE can have important hydrologic consequences, as peak accumulation strongly influences:
- The volume of water available for spring runoff
- The timing of streamflow peaks
- Soil moisture recharge and groundwater contributions

<div style="padding: 10px;">
  <img src="../img/peak_swe_example.png" style="width:85%;">
</div>  

_Example daily SWE at a single site, showing two important periods in snow processes: accumulation (before peak) and ablation (after peak). The vertical line marks peak SWE._

### 5.1 Comparing Modeled and Observed Peak SWE at All Sites in the Watershed
In this section, we evaluate observed and modeled peak SWE for all stations within our watershed and for all years selected in the `StartDate` and `EndDate` above. 

#### 📋 Modeled SWE on the Date of Observed Peak SWE (magnitude)  
This comparison evaluates the modeled SWE on the **specific day when observed SWE reaches its maximum.** By fixing the timing to the observed peak, this comparison isolates errors in SWE magnitude.  
It answers the question: *How much SWE does the model simulate on the day the observed snowpack reaches its maximum?*

In [ ]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_swe_cols = obs_df.columns.tolist()
mod_swe_cols = model_df.columns.tolist()

Use the `modeled_swe_at_observed_peak` utility function to perform the same-day SWE comparison.

In [ ]:
# compute the same-day SWE comparison during the observed peak SWE for each of the observation and modeled sites
df_observed_peak = snow_utils.modeled_swe_at_observed_peak(obs_df, model_df)
df_observed_peak

##### 📊 Visualize the amount of SWE on **the day of observed peak SWE occurs** for both the model and observations at each station

In [ ]:
# Rearrange the dataframe to long format for easier plotting. The function `.melt({})` unpivots the DataFrame from wide to long format.
df_long = (
    df_observed_peak
    .reset_index()  
    .melt(
        id_vars=['Station', 'Water_Year', 'Date'],
        value_vars=['Observed', 'Modeled'],
        var_name='Source',
        value_name='SWE'
    )
)

df_long.head()

In [ ]:
# Create scatter plot of observed and modeled SWE on the day of observed peak SWE
scatter_obs_peak = df_long.hvplot.scatter(
    x='Station',
    y='SWE',
    by='Source',  # Observed vs Modeled
    ylabel='SWE on Observed Peak Day (m)',
    title='Observed and Modeled SWE on the Day of Observed Peak SWE',
    size=70,
    width=700,
    height=450,
    alpha=0.8,
    hover_cols=['Water_Year'],
    rot=45
)

# Customize the scatter plot appearance
scatter_by_station = (
    scatter_obs_peak  
    .opts(legend_position='top_right')
)

scatter_by_station

#### 📋 Modeled vs Observed Peak SWE Comparison (timing & magnitude)  
This comparison evaluates the modeled and observed peak SWE values and their corresponding dates independently. Unlike the previous comparison that fixed the timing to the observed peak swe, this analysis shows the actual days of modeled and observed peak SWE, which may occur on different dates. As a result, it captures errors in both **peak SWE magnitude** and **peak timing**.

In [ ]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_both_peak = snow_utils.modeled_vs_observed_peak_swe(obs_df, model_df)
df_both_peak

##### 📊 Visualize the quantity of peak SWE for both the model and observations at each station

In [ ]:
# Create the scatter plot
scatter_plot_both_peak = df_both_peak.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (m)',
    ylabel='Modeled SWE (m)',
    title='Modeled vs. Observed Peak SWE',
    size=35,
    width=500,
    height=400,
    color='#E69F00'
)#.relabel('Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_both_peak[['Observed', 'Modeled']].max().max()

one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_both_peak * one_to_one_line).opts( #scatter_plot_obs_peak * 
    legend_position='bottom_right'
)

scatter_with_line

### 5.2 Visualizing Model Error for Peak SWE

The previous scatter plots indicate that the modeled and observed peak SWE magnitude and timing don't always align. Next, we plot the degree to which   

The previous scatter plots highlight differences between modeled and observed peak SWE timing and magnitude, but interpreting these variations can be challenging when comparing modeled and observed values directly. To make these differences more explicit, we compute errors in both peak timing and peak SWE magnitude and visualize them directly. This approach clarifies both the direction and magnitude of model bias and facilitates comparison across stations and water years.

First, add columns `Peak_Date_Diff_Days` and `Peak_SWE_Diff` to the DataFrame `df_both_peak` for computed difference in peak SWE date difference and peak SWE quantity between modeled and observed:

In [ ]:
# Compute the difference in peak SWE days and peak SWE amounts between modeled and observed
df_both_peak['Peak_Date_Diff_Days'] = (df_both_peak['Modeled_Date'] - 
                                      df_both_peak['Observed_Date']).dt.days
df_both_peak['Peak_SWE_Diff'] = (df_both_peak['Modeled'] - 
                           df_both_peak['Observed'])

df_both_peak

##### 📊 Visualize the error between the modeled and observed peak SWE  

In [ ]:
# Filter to separate each water year
year1 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].min()]
year2 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].max()]

bar1 = year1.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title=f'Peak SWE Date Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_Date_Diff_Days',
    hover_cols=['Modeled', 'Observed']
)
bar2 = year1.hvplot.bar(
    x='Station',
    y='Peak_SWE_Diff',
    rot=45,
    ylabel='SWE Difference (m)',
    title=f'Peak SWE Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_SWE_Diff',
    hover_cols=['Modeled', 'Observed']
)

# Combine side by side
layout = (bar1 + bar2)
layout.opts(shared_axes=False)

The left panel shows the timing error (date difference) and the right panel the magnitude error (SWE difference). When we computed the difference in date and SWE quantity above, we took `modeled - observed` so:  

| | DATE OF PEAK SWE | PEAK SWE QUANTITY |
|---|---|---|
| + Positive Values | modeled AFTER observed | modeled GREATER THAN observed |
| - Negative Values | modeled BEFORE observed | modeled LESS THAN observed |  

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>Looking at the two plots, what could be some reasons for the model having simulated peak SWE so much earlier and less than observed in Paradise Meadow (PDS)? Perhaps try changing the <code>my_site_code</code> from earlier in the notebook to rerun <code>nwm_utils.comparison_plots()</code> to see the timeseries. 

<br>What happens if you change the year that is plotted? <br>✏️ Try modifying the bar plot code from <code>bar1 = year1.hvplot.bar</code> to <code>bar1 = year2.hvplot.bar</code> </p>
</div>

#### 📊 Next, we combine the timing and magnitude errors and plot them together for each station.

In [ ]:
scatter = df_both_peak.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='Peak_SWE_Diff',
    by='Station', # Water_Year
    xlabel='Peak SWE Timing Error (days)',
    ylabel='Peak SWE Magnitude Error (m)',
    title='Peak SWE Timing vs Magnitude Error',
    size=80,
    width=600,
    height=400,
    hover_cols=['Water_Year']
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='top_left', show_grid=True)


✏️ **Try changing how we view this plot.**  
We can modify a line in the section of code from `by='Station'` to `by='Water_Year'` to better visualize the errors in the different Water Years.  
Are there any patterns that jump out? Which year was modeled peak SWE consistently less than observed peak SWE? 

### 5.3 Visualizing Model Error for Melt Period
The function `compute_melt_period_obs_vs_model` summarizes melt behavior for both the observed and modeled SWE time series at each station and for each water year, allowing the two datasets to be compared side by side. For every station-year combination, it identifies the date of peak SWE, then defines the end of the melt period as the first date after that peak when SWE reaches zero and stays at zero for at least `min_zero_days` consecutive days. Using these two dates, it computes the melt period length in days and the average melt rate over the full melt period, expressed in meters per day. The final output is a table that reports these melt timing and melt rate statistics for both observations and model simulations, making it possible to evaluate whether the model melts snow too early or too late, too quickly or too slowly, and how well it reproduces the overall timing and pace of seasonal snow disappearance across sites and years.


In [ ]:
melt_df_stats = snow_utils.compute_melt_period_obs_vs_model(obs_df, model_df, min_zero_days=10)
melt_df_stats

From a snow hydrologist’s perspective, this table is very useful because it captures three core aspects of seasonal snow behavior: **peak timing and magnitude**, **melt-out timing**, and **average melt rate**. The timeline or dumbbell view is often the most intuitive for timing diagnostics, while the 1:1 scatter is the best overall summary of performance.

In [ ]:
plot_utils.plot_scatter_melt_metrics(melt_df_stats)

While site-level comparisons provide useful detail, it is often more informative to evaluate model behavior at the **process level** to understand systematic tendencies across the watershed. In this step, we assess whether the model tends to melt snow earlier or later than observed by analyzing the difference in melt period length (model minus observed). Positive differences indicate that the model retains snow longer (later melt), while negative differences indicate earlier melt. The following visualization summarizes this behavior across all stations and water years, highlighting both the direction and magnitude of melt timing bias, and providing an overall assessment of whether the model systematically accelerates or delays snowmelt processes.

In [ ]:
# Process-level assessment
plot_utils.plot_melt_bias_summary(
    melt_df_stats,                          # Pandas DataFrame containing site melt statistics
    "Melt_Period_Days_Obs",                 # Name of the column corresponding with the observation data
    "Melt_Period_Days_Mod",                 # Name of the column corresponding with the modeled data
    "Melt Period Length (Obs vs Model)"     # Title of the plot
)

## 6. Compute Statistics and Error Metrics  
The previous section visualized when and where modeled SWE differs from observations, both in terms of peak SWE timing and magnitude. However, visual inspection alone makes it difficult to compare performance across sites or to summarize model behavior in a consistent or quantifiable way. In this section, we compute commonly used statistical error metrics to quantify model performance, allowing us to objectively assess bias, error magnitude, and variability for sites within the watershed.  


### 6.1 Metadata Preparation and Temporal Alignment

First, we need to create a metadata DataFrame that includes at least the following fields, using these exact column names: `site_id`, `site_name`, `latitude`, and `longitude`. In this example, `gdf_in_bbox` already contains the information of interest. Using these specific column names is required to ensure compatibility with the evaluation functions.

In [ ]:
# select proper columns to create a metadata dataframe that can be used for plotting and metric calculations later on.
metadata_df = gdf_in_bbox[['code', 'name', 'latitude', 'longitude']]

# rename columns to ensure compatibility with the evaluation functions
metadata_df = metadata_df.rename(columns={
    'code': 'site_id',
    'name': 'site_name',
    'latitude': 'latitude',
    'longitude': 'longitude'
})

# correct the site_id in the metadata dataframe to match the format of the site_id in the obs and mod dataframes (i.e., add network prefix)
metadata_df['site_id'] = metadata_df['site_id'].apply(lambda x: f"{x}:CA:CCSS")

metadata_df.head()

Then, we need to ensure both datasets (observed and modeled) cover the same time period so that the statistical metrics can be computed consistently.

In [ ]:
obs_df_alinged, model_df_aligned = dataPrep_utils.align_dataframes_on_common_dates(
    obs_df,
    model_df,
    date_col_obs="Date",
    date_col_mod="Date",
    drop_na=False
)

print(obs_df_alinged.shape, model_df_aligned.shape)

### 6.2 Compute Metrics

Now that both dataframes are aligned by time, we can compute the metrics. The following is some examples demonstrating how to use individual metrics of interest for a specific site, referenced as `my_site_code`.

In [ ]:
bias = metric_utils.bias(obs_df_alinged.loc[:, f'{my_site_code}'], model_df_aligned.loc[:, f'{my_site_code}'])
abs_bias = metric_utils.absolute_relative_bias(obs_df_alinged.loc[:, f'{my_site_code}'], model_df_aligned.loc[:, f'{my_site_code}'])
srho = metric_utils.spearman_rank(obs_df_alinged.loc[:, f'{my_site_code}'], model_df_aligned.loc[:, f'{my_site_code}'])

print(f"For site {my_site_code}: bias = {bias}, abs_bias = {abs_bias}, srho = {srho}")

Apply it to all sites using the `calculate_metrics` capability of the evaluation framework.

In [ ]:
metrics_df = evaluation_utils.calculate_metrics(
    obs_df_alinged,                # Pandas DataFrame containing the time series observations.    
    model_df_aligned,              # Pandas DataFrame containing the time series model calculations.
    metadata_df,                   # Pandas DataFrame containing location and site attribute information.
    metrics_list=None,             # List of metrics to calculate. None indicates that all metrics will be computed.
    write_csv=False,               # Indicates the the output should not be saved to CSV (default = False)
    csv_path=None,                 # Indicates the the output should not be saved to CSV (default = None)
)
metrics_df

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    If you recall from earlier, we plotted the timeseries of our selected station. Replot it below. Do the metrics make sense given the visual comparison between modeled and observed? For example, when you look at the timeseries, is the model consistently predicting SWE to be higher or lower than observations? Does this align with the <b>Bias</b> sign (+ or -)?
  </p>
</div> 

The metrics above are computed over the entire snow season, including both accumulation and ablation periods. If desired, you can subset the data and recompute the metrics to examine differences between these phases. For example, if we define the **ablation period** as April through June, the following code computes statistics for data within those months. This approach provides a more detailed understanding of how well the model represents snow accumulation versus melt processes and where performance differences may occur.

In [ ]:
ablation_months = [4, 5, 6]  # 4:April, 5:May, 6:June. 

# Create subsets of the aligned dataframes that exclude the ablation months
obs_df_alinged_ablation = obs_df_alinged[obs_df_alinged.index.month.isin(ablation_months)]
model_df_aligned_ablation = model_df_aligned[model_df_aligned.index.month.isin(ablation_months)]

# Compute metrics for the ablation period
metrics_df_ablation = evaluation_utils.calculate_metrics(
    obs_df_alinged_ablation,
    model_df_aligned_ablation,
    metadata_df,
    metrics_list=None,
    write_csv=False,
    csv_path=None,
)

metrics_df_ablation

For some sites, Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

In [ ]:
metrics_df.hvplot.bar(
    x='site_id',
    y='nse',
    rot=45,
    ylabel='Nash–Sutcliffe Efficiency',
    title='NSE by Station',
    height=400,
    width=600,
    bar_width=0.5
)


In [ ]:
metrics_df.hvplot.scatter(
    x='site_id',
    y='bias',
    size=100,
    rot=45,
    ylabel='Bias (m)',
    title='Mean SWE Bias by Station'
)


### 6.3 Combining Magnitude (Absolute Relative Bias) and Timing (Spearman’s Rho) Using the Condon Metric 

The Condon diagram separates model performance into quadrants based on two metrics: **Spearman’s rho** (shape/time agreement) and **relative bias** (magnitude error). The horizontal line at 0.5 distinguishes whether the model captures the temporal pattern well (above 0.5 = good shape), while the vertical line is traditionally placed at a relative bias of 1.0, which represents a 100% error. This means the model’s total error is as large as the observed signal itself. This threshold has a clear physical interpretation and is used in the original Condon framework to distinguish acceptable vs. large bias. 

In [ ]:
%reload_ext autoreload

In [ ]:
plot_utils.plot_condon_diagram(metrics_df, variable="SWE")